## Übung 02

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

### Teil 1 – Zwei gekoppelte Pendel

**Teil 1 der Übung wird im Wesentlichen in der Vorlesung am Mittwoch (28.10) besprochen.**

Wir betrachten zwei gekoppelte Pendel. Die Auslenkungen werden durch $x_1$ und $x_2$ beschrieben. Die kinetische und die potentielle Energie seien

$$T=\frac12m_1\dot x_1^2+\frac12m_2\dot x_2^2,$$

$$V=\frac12\left[k_1x_1^2+k_2x_2^2+k_{12}(x_1-x_2)^2\right].$$

Dabei beschreiben $k_1$ und $k_2$ die Rückstellkräfte der einzelnen Pendel; $k_{12}$ beschreibt die Kopplung.

### 1a) Hamiltonfunktion und Bewegungsgleichungen

Wie sieht eine Realisierung des Systems mit zwei Massen und Federn aus?

Bestimmen Sie die kanonischen Impulse $p_1$ und $p_2$ und die Hamiltonfunktion

$$H(x_1,x_2,p_1,p_2),$$

und leiten Sie alle vier Hamilton-Gleichungen her.

### 1b) Skalierte kanonische Variablen

Betrachten Sie zunächst einen einzelnen Oszillator mit Masse $m$, Koordinate $x$ und Impuls $p$. Wir führen den neuen Impuls

$$P=\frac{p}{\sqrt m}$$

ein. Wie muss die neue Koordinate $Q$ gewählt werden, damit die Transformation kanonisch beziehungsweise symplektisch ist? **Hinweis:** Eine kanonische Transformation erhält die orientierte Fläche im Phasenraum. 

Schreiben Sie danach die Hamiltonfunktion in den Variablen $(Q,P)$ und vergewissern Sie sich, dass die Bewegungsgleichungen weiterhin die kanonische Form haben:

$$\dot P=-\frac{\partial H}{\partial Q},\qquad \dot Q=\frac{\partial H}{\partial P}.$$

### 1c) Gekoppeltes System in den neuen Variablen

Verwenden Sie für die gekoppelten Pendel eine analoge Transformation

$$P_i=\frac{p_i}{\sqrt{m_i}},\qquad Q_i=\cdots.$$

Schreiben Sie die Hamiltonfunktion des gekoppelten Systems in den Variablen $(Q_1,Q_2,P_1,P_2)$ und leiten Sie daraus die vier Bewegungsgleichungen ab.

### 1d) Numerische Untersuchung

In der Vorlesung werden wir ausgehend von der Form aus c) die Bewegungsgleichungen analytisch weiterbehandeln. Lösen Sie hier die Gleichungen numerisch, um ein Bild von der Physik zu bekommen:

Setzen Sie

$$m_1=m_2=1,\qquad k_1=k_2=1$$

und lösen Sie die Bewegungsgleichungen numerisch für verschiedene Kopplungsstärken $k_{12}$. Verwenden Sie zunächst

$$Q_1(0)=1,\quad Q_2(0)=0,\quad P_1(0)=P_2(0)=0$$

sowie $k_{12}=0;\,0.05;\,0.1$. Stellen Sie $Q_1(t)$ und $Q_2(t)$ dar und beschreiben Sie, wie sich die Bewegung mit der Kopplung verändert.

Vervollständigen Sie dazu die rechte Seite der Differentialgleichung. Die übrige Simulations- und Plot-Infrastruktur ist vorgegeben.

In [ ]:
def coupled_rhs(t, state, m1, m2, k1, k2, k12):
    Q1, Q2, P1, P2 = state
    # TODO: dQ1, dQ2, dP1 und dP2 aus den Hamilton-Gleichungen
    raise NotImplementedError

m1 = m2 = 1.0
k1 = k2 = 1.0
initial_state = np.array([1.0, 0.0, 0.0, 0.0])
couplings = [0.0, 0.05, 0.1]
t_eval = np.linspace(0.0, 80.0, 4001)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True,
                         constrained_layout=True)

solutions = {}
for ax, k12 in zip(axes.flat, couplings):
    solution = solve_ivp(
        coupled_rhs, (t_eval[0], t_eval[-1]), initial_state, t_eval=t_eval,
        args=(m1, m2, k1, k2, k12), rtol=1e-10, atol=1e-12,
    )
    solutions[k12] = solution
    ax.plot(solution.t, solution.y[0], label="$Q_1$")
    ax.plot(solution.t, solution.y[1], label="$Q_2$")
    ax.set_title(rf"$k_{{12}}={k12:g}$")
    ax.grid(alpha=0.25)
    ax.legend()

for ax in axes[-1, :]:
    ax.set_xlabel("Zeit $t$")
for ax in axes[:, 0]:
    ax.set_ylabel("Auslenkung")
plt.show()

Wiederholen Sie den Versuch außerdem für die beiden Anfangsbedingungen

$$Q_1(0)=Q_2(0)=1,\qquad P_1(0)=P_2(0)=0$$

und

$$Q_1(0)=1,\quad Q_2(0)=-1,\qquad P_1(0)=P_2(0)=0.$$

Was ist an diesen beiden Bewegungen besonders?

### Teil 2 – Eine symplektische Mittelpunktregel

Wir betrachten nun den harmonischen Oszillator

$$H(q,p)=\frac12(q^2+p^2),\qquad \dot q=p,\quad \dot p=-q.$$



### 2a) Explizite Mittelpunktregel

Für eine Differentialgleichung $\dot{\boldsymbol y}=f(\boldsymbol y)$ lautet die explizite Mittelpunktregel

$$\boldsymbol y_{n+1/2}^{*}=\boldsymbol y_n+\frac h2f(\boldsymbol y_n),$$

$$\boldsymbol y_{n+1}=\boldsymbol y_n+h f(\boldsymbol y_{n+1/2}^{*}).$$

Leiten Sie für den harmonischen Oszillator die Update-Matrix $M_{\mathrm{expl}}$ mit

$$\begin{pmatrix}q_{n+1}\\p_{n+1}\end{pmatrix}=M_{\mathrm{expl}}\begin{pmatrix}q_n\\p_n\end{pmatrix}$$

her. Berechnen Sie ihre Determinante. Ist die Abbildung symplektisch? Für einen Freiheitsgrad genügt dazu die Bedingung $\det M=1$.

### 2b) Der tatsächliche Mittelpunkt

Ersetzen Sie den explizit geschätzten Mittelpunkt durch den tatsächlichen Mittelpunkt zwischen altem und neuem Zustand:

$$\boldsymbol y_{n+1/2}=\frac{\boldsymbol y_n+\boldsymbol y_{n+1}}2.$$

Zeigen Sie, dass daraus für den Oszillator die gekoppelten Gleichungen

$$q_{n+1}=q_n+\frac h2(p_n+p_{n+1}),$$

$$p_{n+1}=p_n-\frac h2(q_n+q_{n+1})$$

folgen. Lösen Sie dieses lineare Gleichungssystem nach $(q_{n+1},p_{n+1})$, bestimmen Sie die Update-Matrix $M_{\mathrm{impl}}$ und prüfen Sie deren Determinante. 

Vervollständigen Sie den untenstehenden code füur die implizite Mittelpunktsregel.

In [ ]:
def explicit_midpoint(q, p, h):
    q_mid = q + 0.5*h*p
    p_mid = p - 0.5*h*q
    q_new = q + h*p_mid
    p_new = p - h*q_mid
    return q_new, p_new

def implicit_midpoint(q, p, h):
    # TODO: Das lineare 2x2-System für q_new und p_new lösen.
    raise NotImplementedError

def simulate_midpoint(step, q0, p0, h, n_steps):
    q = np.empty(n_steps + 1)
    p = np.empty(n_steps + 1)
    q[0], p[0] = q0, p0
    for n in range(n_steps):
        q[n + 1], p[n + 1] = step(q[n], p[n], h)
    return h*np.arange(n_steps + 1), q, p

### Numerischer Vergleich

Wenn die letzte Aufgabe richtig implementiert ist, löst der code  unten die Bewegungsgleichungen bmit den zwei Methoden, und vergleicht die Trajektorien im Phasenraum und den Energiefehler. Diskutieren Sie das Ergebnis.

In [ ]:
midpoint_methods = {
    "explizit": explicit_midpoint,
    "implizit": implicit_midpoint,
}
q0_mid, p0_mid = 1.0, 0.0
h_mid, T_mid = 0.2, 200.0
n_mid = int(round(T_mid/h_mid))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for name, method in midpoint_methods.items():
    t, q, p = simulate_midpoint(method, q0_mid, p0_mid, h_mid, n_mid)
    energy_error = 0.5*(q**2 + p**2) - 0.5*(q0_mid**2 + p0_mid**2)
    axes[0].plot(q, p, label=name)
    axes[1].plot(t, energy_error, label=name)

axes[0].set(xlabel="$q$", ylabel="$p$", title="Phasenraum")
axes[0].set_aspect("equal", adjustable="box")
axes[1].set(xlabel="Zeit $t$", ylabel="$H_n-H_0$", title="Energiefehler")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.show()